# RL Post-Training using Hugging Face TRL and GRPO

This notebook is based on the TRL GRPO Example, [GRPO Trainer](https://huggingface.co/docs/trl/en/grpo_trainer), to fine-tune a Qwen2.5 0.5B model to answer math questions using the DeepMath-103k dataset and TRL's GRPO (Group Relative Policy Optimization) implementation.

To scale this implementation efficiently across multiple GPUs without changing training logic, we use Ray Train.

This notebook consists of the following steps:
1. [Set up Ray](#set-up)
2. [Quick Summary of GRPO and the DeepMath dataset](#quick-summary)
3. [Running TRL with Ray Train](#training)

Uncomment and run the following line to install all the necessary dependencies. (This notebook is being tested with `transformers==4.57.6`):


In [ ]:
#! pip install "trl[vllm]" "math_verify" "transformers==4.57.6"

(set-up)=

## 1. Set up Ray

Use `ray.init()` to initialize a local cluster. By default, this cluster contains only the machine you are running this notebook on. You can also run this notebook on an Anyscale cluster.


In [ ]:
import ray

ray.init()

Use `ray.cluster_resources()` to check which resources your cluster has access to.
If you're running this notebook on your local machine or Google Colab, you should see the number of CPU cores and GPUs available to you.

In [ ]:
from pprint import pprint

pprint(ray.cluster_resources())

This notebook RL post-trains a pretrained Qwen2.5 model to learn how to answer math questions using the DeepMath dataset and [GRPO](https://arxiv.org/abs/2402.03300).
To parallelize this training efficiently across multiple GPUs and/or nodes, we use Ray Train to easily achieve this with fault-tolerance, observability, and more.

You can change these two variables to control whether the training, which we'll explain later, uses CPUs or GPUs, and how many workers to spawn. Each worker will claim one CPU or GPU so make sure to not request more resources than are available. By default, the training runs with one GPU worker.

In [ ]:
use_gpu = True  # set this to `False` to run on CPUs
num_workers = 4  # set this to the number of GPUs or CPUs you want to use

(quick-summary)=

## 2. Short Summary of the DeepMath dataset and GRPO algorithm

To test and train our model, we use the DeepMath-103k dataset ([He et al., 2025](https://arxiv.org/abs/2504.11456)), consisting of challenging questions across a wide range of mathematical subjects including Algebra, Calculus, Number Theory, Geometry, Probability, and Discrete Mathematics. These questions are more heavily distributed towards Levels 5 to 9 compared to alternative datasets making them highly complex and difficult to solve without specialist training.

The questions consist of a question prompt for the model and a solution in a LaTeX format to provide a consistent format that complex mathematical equations can be expressed.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("trl-lib/DeepMath-103K", split="train").shuffle(seed=123).select(range(10))

pprint(dataset[0])
pprint(dataset[1])

To RL fine-tune our model to answer math questions, we use TRL's GRPO (Group Relative Policy Optimization) implementation that was proposed by [Shao et al., 2024](https://arxiv.org/abs/2402.03300). GRPO is an online algorithm that iteratively improves by training on data the model generates itself, achieved through four steps: generating completions, computing the advantage, estimating the KL divergence, and computing the loss. Importantly, multiple completions are sampled for each question and scored with a reward function, the advantage is then computed by normalising each score relative to the group's own mean and standard deviation, reflecting how much better or worse a completion is compared to its peers.

$$
\mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{\sum_{i=1}^{G} |o_i|} \sum_{i=1}^{G} \sum_{t=1}^{|o_i|} \left[ \frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\left[\pi_\theta(o_{i,t} \mid q, o_{i,<t})\right]_{\text{no grad}}} \hat{A}_{i,t} - \beta \mathbb{D}_{\text{KL}}\left[\pi_\theta \| \pi_{\text{ref}}\right] \right],
$$

For a deeper summary of the algorithm, the [TRL GRPO page](https://huggingface.co/docs/trl/en/grpo_trainer#looking-deeper-into-the-grpo-method) provides a more detailed description.

(training)=

## 3. Using Hugging Face TRL (Transformer Reinforcement Learning) with Ray Train

In comparison to supervised pre-training where models are optimised to minimise the error for their next token, RL-based post-training aims to maximise their reward from a prompt. Therefore, it's crucial to define a reward function to measure the success and to train a model.

In this notebook, we use the `trl.rewards.accuracy_reward` function to check whether the model's answer matches the dataset's answer. Because the answers are written in LaTeX, both responses must be parsed before they can be compared. The default `trl.rewards.accuracy_reward` implementation applies timeouts to the `parse` and `verify` functions, which are incompatible with Ray. The version defined below disables these timeouts by setting `parsing_timeout=0` and `timeout_seconds=0`.

In [ ]:
from latex2sympy2_extended import NormalizationConfig
from math_verify import LatexExtractionConfig, parse, verify

def accuracy_reward(completions, solution, **kwargs):
    """Reward function that checks mathematical accuracy.

    This is a copy of `trl.rewards.accuracy_reward`.
    The only difference is `parse(..., parsing_timeout=0)` and `verify(..., timeout_seconds=0)`
    to avoid `signal.alarm()` issues with ray.
    """
    contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, sol in zip(contents, solution, strict=True):
        gold_parsed = parse(sol, parsing_timeout=0)
        if len(gold_parsed) != 0:
            # We require the answer to be provided in correct latex (no malformed operators)
            answer_parsed = parse(
                content,
                extraction_config=[
                    LatexExtractionConfig(
                        normalization_config=NormalizationConfig(units=True),
                        # Ensures that boxed is tried first
                        boxed_match_priority=0,
                        try_extract_without_anchor=False,
                    )
                ],
                extraction_mode="first_match",
                parsing_timeout=0,
            )
            reward = float(verify(gold_parsed, answer_parsed, timeout_seconds=0))
        else:
            # If the gold solution cannot be parsed, we assign `None` to skip this example
            reward = None
        rewards.append(reward)

    return rewards

Now, we need to build our training function that is parallelized to all the workers. We build a `GRPOTrainer` which implements the GRPO algorithm.

We use `vllm_mode="colocate"`, where each worker runs a vLLM instance that consumes about 30% of the GPU memory. The alternative `"server"` mode dedicates one worker to generation with vLLM while the others perform learning, but this introduces inter-GPU communication overhead that can reduce throughput. See [this blog post](https://huggingface.co/blog/vllm-colocate) for more details.

Two Ray Train-specific additions integrate the HF Trainer into the Ray Train ecosystem:

- **`RayTrainReportCallback`** — hooks into the HF Trainer's logging to forward metrics and checkpoints to Ray Train after each training step. This is what populates the `Result` object returned by `trainer.fit()` with metrics like `reward` and the best checkpoint.
- **`prepare_trainer`** — configures the HF Trainer for distributed execution across Ray workers. It disables HF's built-in distributed setup so that Ray Train controls the process group instead, and ensures correct device placement on each worker.

In [ ]:
from ray.train.huggingface.transformers import RayTrainReportCallback, prepare_trainer
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset

def train_func(config):
    # Datasets
    dataset = load_dataset("trl-lib/DeepMath-103K", split="train").shuffle().select(range(100))


    # Use vllm_mode="colocate" to allow easy scaling as each GPU handles their own vLLM instance
    training_args = GRPOConfig(
        per_device_train_batch_size=4, 
        use_vllm=True, vllm_mode="colocate", 
        vllm_gpu_memory_utilization=0.3,
        num_train_epochs=2,
    )

    # GRPO Trainer
    trainer = GRPOTrainer(
        model="Qwen/Qwen2.5-0.5B",
        args=training_args,
        reward_funcs=accuracy_reward,
        train_dataset=dataset,
    )

    # Report metrics and checkpoints to Ray Train
    trainer.add_callback(RayTrainReportCallback())

    # Prepare your trainer for Ray Data integration
    trainer = prepare_trainer(trainer)

    # Start Training
    trainer.train()

With your `train_func` complete, you can now instantiate the {class}`~ray.train.torch.TorchTrainer`. Aside from calling the function, set the `scaling_config`, which controls the number of workers and resources used, and the `run_config` to configure checkpointing.

The `CheckpointConfig` controls how Ray Train saves and selects checkpoints during training:

- **`num_to_keep=1`** — only the single best checkpoint is retained on disk, saving storage.
- **`checkpoint_score_attribute="reward"`** — checkpoints are ranked by the `"reward"` metric, which is the mean reward across the batch as reported by `RayTrainReportCallback`.
- **`checkpoint_score_order="max"`** — higher reward is better, so Ray Train keeps the checkpoint with the highest reward seen across all training steps.

In [ ]:
from ray.train.torch import TorchTrainer
from ray.train import RunConfig, ScalingConfig, CheckpointConfig

trainer = TorchTrainer(
    train_func,
    scaling_config=ScalingConfig(num_workers=num_workers, use_gpu=use_gpu),
    run_config=RunConfig(
        checkpoint_config=CheckpointConfig(
            num_to_keep=1,
            checkpoint_score_attribute="reward",
            checkpoint_score_order="max",
        ),
    ),
)

Finally, call the `fit` method to start training with Ray Train. Save the `Result` object to a variable so you can access metrics and checkpoints.

In [ ]:
result = trainer.fit()

You can use the returned `Result` object to access metrics and the Ray Train `Checkpoint` associated with the last iteration.

In [ ]:
result

## Summary

This notebook demonstrated how to use Ray Train to scale RL post-training of a Qwen2.5 0.5B model across multiple GPUs using TRL's GRPO implementation and the DeepMath-103k dataset.

The key Ray Train integration points were:

- **`RayTrainReportCallback`** — forwards HF Trainer metrics and checkpoints to Ray Train after each step.
- **`prepare_trainer`** — configures the HF Trainer to run distributed training under Ray's process group.
- **`TorchTrainer`** — launches `train_func` on all workers with the configured resources.
- **`CheckpointConfig`** — automatically tracks and retains the best checkpoint based on the `"reward"` metric.

From here, you can:
- Scale to more GPUs or multiple nodes by increasing `num_workers` in `ScalingConfig`.
- Swap in a different base model or dataset by updating the `model` argument and `load_dataset` call.
- Experiment with different reward functions to train for other tasks.

## See also

* {doc}`Ray Train Examples <../../examples>` for more use cases
* {ref}`Ray Train User Guides <train-user-guides>` for how-to guides